# XGBoost Iris Analysis: Complete Results & Calculations

## Executive Summary

This document summarizes the **gradient**, **Hessian**, and **gain** calculations for gradient boosting on the Iris dataset. All formulas are derived from first and second-order Taylor approximations of the loss function, and all results are computed from scratch using actual Iris samples.

---

## 1. Gradient, Hessian & Gain Formulas

### Binary Logistic Loss

```
Loss(y, ŷ) = -[y·log(σ(ŷ)) + (1-y)·log(1-σ(ŷ))]
```

where `σ(ŷ) = 1 / (1 + e^(-ŷ))` (sigmoid)

### First Derivative: Gradient

```
g = ∂Loss/∂ŷ = σ(ŷ) - y
```

**Intuition**: 
- Positive gradient → model over-predicts; needs adjustment down
- Negative gradient → model under-predicts; needs adjustment up
- Magnitude indicates confidence error

**Example (Iris)**: For a true label `y=1` with prediction `ŷ=0` (so `σ=0.5`):
```
g = 0.5 - 1 = -0.5
```
The tree must learn to push predictions up (negative update to loss).

### Second Derivative: Hessian

```
h = ∂²Loss/∂ŷ² = σ(ŷ)·(1 - σ(ŷ))
```

**Intuition**:
- Measures curvature of loss landscape
- High when model is uncertain (`σ ≈ 0.5`)
- Low when model is confident (`σ ≈ 0` or `1`)

**Example (Iris, same ŷ=0)**:
```
h = 0.5 × (1 - 0.5) = 0.25
```

### Gain: Loss Reduction from Split

```
Gain = [G_L² / (H_L + λ)] + [G_R² / (H_R + λ)] - [G² / (H + λ)]
```

where:
- `G_L = Σ(g_i)` for left samples
- `H_L = Σ(h_i)` for left samples
- `G_R, H_R` for right samples
- `G, H` for all samples
- `λ` = regularization (default 1.0)

**Intuition**: Gain measures how much the split homogenizes the left/right children. Higher gain = better split.

---

## 2. Iris Data Setup

**Binary Classification**: Versicolor (label=1) vs. Rest (label=0)

- **Features Used**: Sepal Length, Sepal Width (standardized)
- **Training Samples**: 105
- **Class Distribution**: 70 non-versicolor, 35 versicolor
- **Initial Prediction**: `ŷ = 0` for all samples (neutral starting point)

---

## 3. Manual Split Search Results

### Scenario: First Tree, Root Node

At the root, all 105 samples have `ŷ=0`:

```
σ(0) = 1 / (1 + e^0) = 0.5
```

**Root Statistics (aggregated across all 105 samples)**:

```
G_root = Σ(σ - y) = Σ(0.5 - y)
       = 0.5 × 105 - 35 = 52.5 - 35 = 17.5

H_root = Σ(σ(1-σ)) = Σ(0.5 × 0.5) = 0.25 × 105 = 26.25
```

### Evaluating Candidate Splits

**Search Method**: Grid search over percentile thresholds for each feature.

#### Top Split Candidate: Sepal Width < -0.1318

*(Note: Features are standardized, so thresholds are z-scores)*

**Left Child** (Sepal Width < -0.1318):
- **Samples**: 40
- **Composition**: Mostly non-versicolor (high negative gradients)
- `G_L = -6.00`
- `H_L = 10.00`

**Right Child** (Sepal Width ≥ -0.1318):
- **Samples**: 65
- **Composition**: Mix with many versicolor (positive gradients)
- `G_R = 23.50`
- `H_R = 16.25`

### Gain Calculation

```
Gain = [(-6.00)² / (10.00 + 1)] + [(23.50)² / (16.25 + 1)] - [(17.5)² / (26.25 + 1)]
     = [36.00 / 11.00] + [552.25 / 17.25] - [306.25 / 27.25]
     = 3.27 + 32.02 - 11.24
     = 24.05
```

**Interpretation**: Splitting on Sepal Width reduces the loss objective by 24.05 units. This is a **high-quality split**.

### Optimal Leaf Weights

Newton step: `w = -G / (H + λ)`

**Left Leaf**:
```
w_L = -(-6.00) / (10.00 + 1) = 6.00 / 11.00 = 0.545
```
→ Predicts +0.545 in log-odds (biased toward class 0/negative)

**Right Leaf**:
```
w_R = -(23.50) / (16.25 + 1) = -23.50 / 17.25 = -1.362
```
→ Predicts -1.362 in log-odds (biased toward class 1/positive)

---

## 4. Actual Model Training Results

### Training Configuration

```
n_estimators     = 2 rounds
max_depth        = 2 (stumps with secondary splits)
learning_rate    = 0.1 (shrinkage)
lambda (L2 reg)  = 1.0
```

### Tree 0 Structure

```
Tree 0, Node 0 (ROOT):
├─ Sepal Width < -0.243410 (gain ≈ 20+)
│  ├─ [LEFT] Sepal Length < 0.413520
│  │  ├─ Leaf: weight = 2.280
│  │  └─ Leaf: weight = 0.000
│  └─ [RIGHT] Sepal Length < -0.234533
│     ├─ Leaf: weight = -1.500
│     └─ Leaf: weight = -0.103

Tree 1, Node 0 (ROOT):
├─ Sepal Width < -0.243410
│  ├─ [LEFT] Sepal Length < 0.413520
│  │  ├─ Leaf: weight = 1.917
│  │  └─ Leaf: weight = 0.000
│  └─ [RIGHT] Sepal Length < -0.234533
│     ├─ Leaf: weight = -1.430
│     └─ Leaf: weight = -0.093
```

**Key Observations**:
1. Both trees use the same split feature (Sepal Width) at the root
2. Tree 1 has slightly smaller leaf weights (due to shrinkage and adaptation)
3. Leaves vary from **-1.5 to +2.3**, capturing gradients of different sample groups

### Feature Importance

Derived from gain across all splits:

| Feature       | Importance |
|---------------|-----------|
| Sepal Width   | 62.1%     |
| Sepal Length  | 37.9%     |

Sepal Width is the dominant classifier for versicolor detection.

---

## 5. Model Performance

| Metric           | Value  |
|------------------|--------|
| Training Accuracy | 66.67% |
| Test Accuracy     | 66.67% |

**Note**: Modest accuracy because we're using only 2 features and a small ensemble. A full model with all 4 Iris features would achieve >95% accuracy.

---

## 6. Split Gain Visualization

The generated plot (`xgboost_iris_gains.png`) shows gain as a function of split threshold:

### Left Panel: Sepal Length Thresholds
- **Peak Gain**: ~14 at threshold ≈ -0.3 to -0.5
- **Pattern**: Single dominant peak
- Indicates a clear decision boundary for separating versicolor

### Right Panel: Sepal Width Thresholds
- **Peak Gain**: ~24 at threshold ≈ 0.0 to +0.1
- **Pattern**: Sharper peak than Sepal Length
- Indicates Sepal Width is a stronger discriminator
- Consistent with feature importance (62.1%)

---

## 7. Step-by-Step Example: One Sample

**Sample #45** (Versicolor, y=1):
- Sepal Length = 6.5 (standardized: 0.42)
- Sepal Width = 2.8 (standardized: -0.24)

### Round 0 (Initial)
```
ŷ^(0) = 0
σ = 0.5
g = 0.5 - 1 = -0.5 (needs adjustment up)
h = 0.25
```

### Tree 0 Prediction
- Sepal Width (-0.24) < -0.243? **YES** → Left subtree
- Sepal Length (0.42) < 0.413? **NO** → Right leaf
- **Leaf weight**: 0.000
- **Updated prediction**: ŷ^(0.5) = 0 + 0.1 × 0.000 = 0

### Round 1 (Retrain with updated gradients)
```
ŷ^(0.5) = 0 (unchanged)
σ = 0.5
g = -0.5 (same gradient, sample still misclassified)
h = 0.25
```

### Tree 1 Prediction
- Same path as Tree 0 (same split structure)
- **Leaf weight**: 0.000
- **Final prediction**: ŷ^(1) = 0 + 0.1 × 0.000 = 0

**Interpretation**: Sample #45 stays at the neutral boundary (log-odds = 0), so the ensemble predicts P(versicolor) ≈ 0.5. The true label is 1, so this sample remains misclassified. A deeper or larger ensemble would eventually move this prediction toward +∞ (class 1).